In [1]:
# =============================================================================
#  Iris Classification with PyTorch + MLflow Model Registry
# =============================================================================

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
from mlflow.tracking import MlflowClient
import pandas as pd

C:\Users\rasel\anaconda3\envs\DL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ────────────────────────────────────────────────
#  1. Configuration & Tracking Setup
# ────────────────────────────────────────────────

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Iris_PyTorch_Classification")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# ────────────────────────────────────────────────
#  2. Data Preparation
# ────────────────────────────────────────────────

iris = load_iris()
X = iris.data.astype(np.float32)
y = iris.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ── to PyTorch tensors ───────────────────────────────────────
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)
test_dataset  = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [4]:
# ────────────────────────────────────────────────
#  3. Model Definition
# ────────────────────────────────────────────────

class IrisNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)           # raw logits
        return x

model = IrisNet().to(DEVICE)

In [5]:
# ────────────────────────────────────────────────
#  4. Training function
# ────────────────────────────────────────────────

def train_model(model, train_loader, epochs=80, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * Xb.size(0)

        if (epoch+1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}   loss: {running_loss/len(train_loader.dataset):.4f}")

    return model

In [6]:
# ────────────────────────────────────────────────
#  5. Evaluation function
# ────────────────────────────────────────────────

@torch.no_grad()
def evaluate_model(model, test_loader):
    model.eval()
    preds = []
    trues = []

    for Xb, yb in test_loader:
        Xb = Xb.to(DEVICE)
        logits = model(Xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        preds.extend(pred)
        trues.extend(yb.numpy())

    acc       = accuracy_score(trues, preds)
    f1        = f1_score(trues, preds, average="weighted")
    precision = precision_score(trues, preds, average="weighted")
    recall    = recall_score(trues, preds, average="weighted")

    return acc, f1, precision, recall, np.array(preds)

In [7]:
# ────────────────────────────────────────────────
#  6. MLflow Logging + Model Registry
# ────────────────────────────────────────────────

from mlflow.models import infer_signature

with mlflow.start_run(run_name="pytorch-iris-run-003") as run:   # changed run name to distinguish

    # ── Hyperparameters ───────────────────────────────
    epochs = 80
    lr     = 0.01
    hidden = 16

    mlflow.log_params({
        "epochs": epochs,
        "learning_rate": lr,
        "hidden_dim": hidden,
        "optimizer": "Adam",
        "criterion": "CrossEntropyLoss",
        "batch_size": 32,
        "architecture": "3-layer MLP",
        "device": str(DEVICE)
    })

    # Train the model
    print("Training started...")
    trained_model = train_model(model, train_loader, epochs=epochs, lr=lr)
    print("Training completed.")

    # Evaluate
    accuracy, f1, precision, recall, predictions = evaluate_model(trained_model, test_loader)

    print(f"\nTest  accuracy  = {accuracy:.4f}")
    print(f"      f1        = {f1:.4f}")
    print(f"      precision = {precision:.4f}")
    print(f"      recall    = {recall:.4f}")

    # Log evaluation metrics
    mlflow.log_metrics({
        "accuracy":  accuracy,
        "f1_score":  f1,
        "precision": precision,
        "recall":    recall
    })

    # ── Infer model signature (highly recommended) ────────────────────────────────
    try:
        signature = infer_signature(X_test, predictions)
        print("Signature inferred successfully.")
    except Exception as e:
        print(f"Warning: Could not infer signature → {e}")
        signature = None

    # Optional: log a few example predictions for debugging
    mlflow.log_dict(
        {
            "sample_inputs": X_test[:5].tolist(),
            "sample_predictions": predictions[:5].tolist()
        },
        "sample_predictions.json"
    )

    # ── Log and register the PyTorch model ─────────────────────────────
    print("Logging model to MLflow...")

    mlflow.pytorch.log_model(
        pytorch_model       = trained_model,           # explicit keyword for clarity
        artifact_path       = "pytorch_model",
        registered_model_name = "IrisPyTorchModel",
        signature           = signature,
        input_example       = X_test[:5].astype(np.float32),
        # Optional: pin dependency versions if you have reproducibility issues
        # pip_requirements    = ["torch>=2.0.0", "numpy", "scikit-learn"],
        # metadata            = {"trained_on": str(DEVICE), "run_name": run.data.tags.get("mlflow.runName")}
    )

    print(f"Run ID: {run.info.run_id}")
    print("Model successfully logged and registered → IrisPyTorchModel")
    print(f"→ View run: http://127.0.0.1:5000/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

Training started...
Epoch  20/80   loss: 0.0872
Epoch  40/80   loss: 0.0669
Epoch  60/80   loss: 0.0805


2026/03/14 15:04:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 15:04:21 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch  80/80   loss: 0.0595
Training completed.

Test  accuracy  = 0.9778
      f1        = 0.9778
      precision = 0.9792
      recall    = 0.9778
Signature inferred successfully.
Logging model to MLflow...


2026/03/14 15:04:21 WARNING mlflow.utils.requirements_utils: Found torch version (2.7.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.7.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/14 15:04:29 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.22.1+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.22.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Registered model 'IrisPyTorchModel' already exists. Creating a new version of this model...
2026/03/14 15:04:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model nam

Run ID: 96d29bb7d0d643239c65c30e5d15c109
Model successfully logged and registered → IrisPyTorchModel
→ View run: http://127.0.0.1:5000/#/experiments/4/runs/96d29bb7d0d643239c65c30e5d15c109
🏃 View run pytorch-iris-run-003 at: http://127.0.0.1:5000/#/experiments/4/runs/96d29bb7d0d643239c65c30e5d15c109
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


Created version '4' of model 'IrisPyTorchModel'.


In [8]:
# ────────────────────────────────────────────────
#  7. Promote latest version using alias (instead of stage)
# ────────────────────────────────────────────────

client = MlflowClient()

model_name = "IrisPyTorchModel"  # ← define once here

# Get all versions and find the latest (highest version number)
versions = client.search_model_versions(f"name='{model_name}'")

if not versions:
    print("No versions registered yet.")
else:
    # Sort descending by version number (string → int conversion)
    latest_version_obj = max(versions, key=lambda v: int(v.version))
    latest_version = latest_version_obj.version

    # Set alias (overwrites if already exists on another version)
    client.set_registered_model_alias(
        name    = model_name,
        alias   = "champion",           # convention: production / best model
        version = latest_version
    )

    # Optional: also set a "staging" alias if you want separate pre-prod pointer
    # client.set_registered_model_alias(model_name, "staging", latest_version)

    print(f"Version {latest_version} of '{model_name}' now has alias 'champion' (production-ready)")

Version 4 of 'IrisPyTorchModel' now has alias 'champion' (production-ready)


In [9]:
# ────────────────────────────────────────────────
#  8. Compare versions (simple table) — now using aliases instead of stage
# ────────────────────────────────────────────────

versions = client.search_model_versions(f"name='{model_name}'")

data = []
for v in versions:
    run = client.get_run(v.run_id)
    metrics = run.data.metrics

    # Aliases are now in v.aliases (list of strings)
    aliases_str = ", ".join(v.aliases) if v.aliases else "none"

    data.append({
        "version": v.version,
        "aliases": aliases_str,
        "run_id":  v.run_id[:8],
        "accuracy": metrics.get("accuracy", None),
        "creation_time": v.creation_timestamp   # optional: for sorting
    })

df = pd.DataFrame(data)
print("\nRegistered model versions:\n")
print(df.sort_values("version", ascending=False))

# Find best by accuracy (or you could check which has "champion" alias)
best_row = df.loc[df["accuracy"].idxmax()]
best_version = best_row["version"]
best_aliases = best_row["aliases"]

print(f"\nBest version by accuracy: {best_version}  (aliases: {best_aliases})")


Registered model versions:

  version aliases    run_id  accuracy  creation_time
0       4    none  96d29bb7  0.977778  1773497070188
1       3    none  3b23a14f  0.977778  1773495792879
2       2    none  86a7af26  0.977778  1773487899310
3       1    none  82c8403e  0.933333  1773487557688

Best version by accuracy: 4  (aliases: none)


In [10]:
# ────────────────────────────────────────────────
#  9. Load model from registry using alias and predict
# ────────────────────────────────────────────────

# Use @alias syntax — no need to know the version number!
model_uri = f"models:/{model_name}@champion"

loaded_model = mlflow.pytorch.load_model(model_uri)

# Example inference (same as before)
loaded_model.eval()
with torch.no_grad():
    sample = torch.from_numpy(X_test[:8]).to(DEVICE)
    logits = loaded_model(sample)
    probs  = torch.softmax(logits, dim=1)
    preds  = torch.argmax(probs, dim=1).cpu().numpy()

print("\nSample predictions (first 8 test samples) using @champion:")
print("Predicted:", preds)
print("True     :", y_test[:8])


Sample predictions (first 8 test samples) using @champion:
Predicted: [2 1 1 1 2 2 1 1]
True     : [2 1 2 1 2 2 1 1]
